# Radar Pulses — what each waveform is for

Three waveforms, each shown twice: once as a shape you can take apart, and once inside the physical situation that makes it the right answer. The point is not that one is better. Each is the best available answer to a *different* question, and the fastest way to see that is to put each one in front of a problem the others fail.

$$\text{energy}\propto P\tau
\qquad\qquad
\Delta R=\frac{c}{2B}
\qquad\qquad
\Delta v=\frac{\lambda}{2T}$$

Those three quantities are the whole argument. Detection wants the first, range resolution the second, velocity resolution the third — and $\tau$, $B$ and $T$ cannot all be chosen freely. A waveform is a decision about which of them you are willing to spend.

| you need | reach for | because |
|---|---|---|
| to measure **speed** precisely | a long coherent burst of plain pulses | $\Delta v=\lambda/2T$ — only dwell time buys Doppler |
| to **separate two targets in range** at long range | LFM chirp | bandwidth without shortening the pulse |
| fine range on **simple hardware** | phase code | biphase modulation only, no linear sweep |
| to see a **small target beside a big one** | chirp with a weighted filter | the phase code's sidelobe floor is fixed and cannot be weighted away |

Every section runs in time. Press ▶ or drag the slider.

In [1]:
%matplotlib inline
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import ipywidgets as widgets
from IPython.display import display

BG, PANEL, FG = "#05070b", "#0a0d14", "#c9cfda"
MUTED, GRIDC = "#6b7280", "#1b2130"
BLUE, ORANGE, CYAN, RED, GREEN = "#5aa9e6", "#e08a3c", "#3fd0c9", "#e0555c", "#7ddc7d"
FIELD = LinearSegmentedColormap.from_list("rp_field", [
    (0.00, "#eaf3ff"), (0.16, "#5aa9e6"), (0.44, "#0a1622"), (0.50, "#05070b"),
    (0.56, "#211307"), (0.84, "#e08a3c"), (1.00, "#fff3e2")])
HOT = LinearSegmentedColormap.from_list("rp_hot", [
    (0.00, "#05070b"), (0.35, "#123049"), (0.65, "#3fd0c9"),
    (0.85, "#e0c76a"), (1.00, "#fff3e2")])

plt.rcParams.update({
    "figure.dpi": 110, "font.size": 8.5, "axes.titlesize": 9,
    "figure.facecolor": BG, "savefig.facecolor": BG, "axes.facecolor": PANEL,
    "axes.edgecolor": GRIDC, "axes.labelcolor": FG, "text.color": FG,
    "xtick.color": MUTED, "ytick.color": MUTED, "grid.color": GRIDC,
    "axes.grid": True, "grid.alpha": 0.25,
    "axes.spines.top": False, "axes.spines.right": False,
    "legend.facecolor": PANEL, "legend.edgecolor": GRIDC, "legend.framealpha": 0.9,
})

SL = {"style": {"description_width": "96px"},
      "layout": widgets.Layout(width="280px"), "continuous_update": False}
C0, LAM = 2.998e8, 0.03


def panel(ax, edge=None, lw=1.4):
    ax.set_facecolor(PANEL)
    for s in ax.spines.values():
        s.set_visible(True); s.set_color(edge or GRIDC)
        s.set_linewidth(lw if edge else 0.8)
    ax.tick_params(colors=MUTED, labelsize=7)
    return ax


def readout(fig, x, y, lines, color=FG, size=7.4):
    fig.text(x, y, "\n".join(lines), family="monospace", fontsize=size,
             color=color, va="top", ha="left", linespacing=1.55)


def footer(fig, text):
    fig.text(0.010, 0.012, text, family="monospace", fontsize=6.6, color=MUTED)
    fig.text(0.990, 0.012, "radar pulses", family="monospace", fontsize=6.6,
             color=MUTED, ha="right")


def dbv(x, floor=-60.0):
    x = np.abs(np.asarray(x)).astype(float)
    m = x.max()
    if not np.isfinite(m) or m <= 0:
        return np.full(x.shape, floor)
    with np.errstate(divide="ignore"):
        return np.clip(20 * np.log10(np.maximum(x, 1e-300) / m), floor, 0.0)


def timeline(n, step=1, interval=110, desc="time"):
    p = widgets.Play(value=0, min=0, max=n, step=step, interval=interval)
    s = widgets.IntSlider(value=0, min=0, max=n, step=step, description=desc + ":",
                          continuous_update=False,
                          style={"description_width": "96px"},
                          layout=widgets.Layout(width="430px"))
    widgets.jslink((p, "value"), (s, "value"))
    return p, s


BARKER13 = np.array([1, 1, 1, 1, 1, -1, -1, 1, 1, -1, 1, -1, 1], float)


def wf_cw(tau, fs):
    return np.ones(max(int(tau * fs), 8), complex)


def wf_chirp(tau, fs, B):
    n = max(int(tau * fs), 8)
    t = np.arange(n) / fs - tau / 2
    return np.exp(1j * np.pi * (B / tau) * t ** 2)


def wf_barker(tau, fs, code=BARKER13):
    n = max(int(tau * fs), len(code))
    sps = max(n // len(code), 1)
    return np.repeat(code, sps).astype(complex)


def compress(rx, ref, weight="unweighted"):
    w = np.hamming(len(ref)) if weight == "hamming" else np.ones(len(ref))
    return np.convolve(rx, np.conj(ref[::-1]) * w[::-1], "same")


def echo(nsamp, ref, delay_s, fs, amp=1.0, fd=0.0):
    """Place one echo at a delay, with a Doppler phase ramp across the pulse."""
    out = np.zeros(nsamp, complex)
    d = int(round(delay_s * fs))
    n = len(ref)
    if 0 <= d < nsamp - n:
        t = np.arange(n) / fs
        out[d:d + n] = amp * ref * np.exp(2j * np.pi * fd * t)
    return out


print(f"waveform library ready   c = {C0:.4e} m/s   λ = {LAM*100:.1f} cm")
print(f"range resolution c/2B    velocity resolution λ/2T    Barker-13 PSL "
      f"{20*np.log10(1/13):.2f} dB")

waveform library ready   c = 2.9980e+08 m/s   λ = 3.0 cm
range resolution c/2B    velocity resolution λ/2T    Barker-13 PSL -22.28 dB


## The unmodulated pulse — nothing but on and off

The simplest thing a transmitter can do: switch on for $\tau$ seconds at a single frequency. Its spectrum is forced by that rectangle alone,

$$s(t)=\text{rect}(t/\tau)\;\longrightarrow\;S(f)=\tau\,\text{sinc}(f\tau),
\qquad B\approx\frac{1}{\tau}$$

so **duration and bandwidth are locked together**, and that single fact determines everything else. A 10 µs pulse has 100 kHz of bandwidth and therefore 1.5 km of range resolution. Wanting finer range means a shorter pulse, which means less energy on the target, which means less range. There is no third option — that is precisely the deadlock the other two waveforms exist to break.

The matched filter output is the pulse's autocorrelation: a triangle $2\tau$ wide at the base, with no sidelobes at all. That is the one thing the unmodulated pulse does better than anything else — its range response is *clean*, with nothing to mask a neighbouring target.

Watch the two panels move together as you drag $\tau$. Long pulse: narrow spectrum, fat triangle, poor range resolution, good energy. Short pulse: wide spectrum, sharp triangle, fine range resolution, little energy. One slider, and it moves both.

In [2]:
def draw_cw(tau_us, fs_mhz, prf_khz):
    fs = fs_mhz * 1e6
    tau = tau_us * 1e-6
    s = wf_cw(tau, fs)
    n = len(s)
    t = np.arange(n) / fs * 1e6
    B = 1.0 / tau

    fig = plt.figure(figsize=(13.2, 4.6))
    gs = fig.add_gridspec(1, 4, width_ratios=[1, 1, 1, 0.55], wspace=0.3,
                          left=0.05, right=0.995, top=0.86, bottom=0.16)

    a0 = panel(fig.add_subplot(gs[0]), BLUE)
    tt = np.linspace(-0.35 * tau_us, 1.35 * tau_us, 1400)
    car = np.where((tt >= 0) & (tt <= tau_us),
                   np.cos(2 * np.pi * 6 / tau_us * tt), 0.0)
    a0.plot(tt, car, color=BLUE, lw=0.8)
    a0.plot(tt, ((tt >= 0) & (tt <= tau_us)).astype(float), color="#f2f5fa",
            lw=1.2, ls="--")
    a0.set_xlabel("time  (µs)"); a0.set_ylabel("amplitude")
    a0.set_title(f"the pulse — τ = {tau_us:.2f} µs")

    a1 = panel(fig.add_subplot(gs[1]), ORANGE)
    nf = 1 << 14
    S = np.abs(np.fft.fftshift(np.fft.fft(s, nf)))
    f = np.fft.fftshift(np.fft.fftfreq(nf, 1 / fs)) / 1e6
    a1.plot(f, dbv(S), color=ORANGE, lw=1.0)
    a1.axvline(B / 2e6, color=CYAN, lw=0.9, ls=":")
    a1.axvline(-B / 2e6, color=CYAN, lw=0.9, ls=":")
    a1.set_xlim(-5 * B / 1e6, 5 * B / 1e6); a1.set_ylim(-50, 4)
    a1.set_xlabel("frequency  (MHz)"); a1.set_ylabel("level  (dB)")
    a1.set_title(f"spectrum — B ≈ 1/τ = {B/1e6:.3f} MHz")

    a2 = panel(fig.add_subplot(gs[2]), GREEN)
    ac = compress(np.pad(s, n), s)
    rr = (np.arange(len(ac)) - len(ac) / 2) / fs * C0 / 2
    a2.plot(rr, dbv(ac), color=GREEN, lw=1.1)
    a2.axhline(-3, color=GRIDC, lw=0.8, ls=":")
    a2.set_xlim(-2.2 * C0 * tau / 2, 2.2 * C0 * tau / 2); a2.set_ylim(-45, 4)
    a2.set_xlabel("range offset  (m)"); a2.set_ylabel("compressed  (dB)")
    a2.set_title("matched filter output — a clean triangle")

    readout(fig, 0.845, 0.86, [
        "UNMODULATED PULSE", "─" * 26,
        f"τ           {tau_us:>10.2f}µs",
        f"bandwidth   {B/1e6:>10.3f}MHz",
        f"time-bw     {B*tau:>10.2f}",
        f"PRF         {prf_khz:>10.2f}kHz",
        f"duty        {100*tau*prf_khz*1e3:>10.3f}%",
        "", "CONSEQUENCES", "─" * 26,
        f"range res   {C0/(2*B):>10.1f}m",
        f"pulse len   {C0*tau/2:>10.1f}m",
        f"R unamb     {C0/(2*prf_khz*1e3)/1000:>10.2f}km",
        f"comp gain   {10*np.log10(max(B*tau,1)):>10.2f}dB",
        "", "no sidelobes",
        "no compression",
        "τ and B are one knob",
    ])
    footer(fig, f"unmodulated pulse   τ={tau_us:.2f} µs   B=1/τ={B/1e6:.3f} MHz   "
                f"ΔR = c/2B = {C0/(2*B):.0f} m")
    plt.show()


wC = dict(tau_us=widgets.FloatSlider(value=5, min=0.2, max=40, step=0.2,
                                     description="pulse τ (µs):", **SL),
          fs_mhz=widgets.FloatSlider(value=40, min=10, max=100, step=5,
                                     description="sample rate:", **SL),
          prf_khz=widgets.FloatSlider(value=5, min=0.5, max=30, step=0.5,
                                      description="PRF (kHz):", **SL))
display(widgets.HBox([wC["tau_us"], wC["fs_mhz"], wC["prf_khz"]]),
        widgets.interactive_output(draw_cw, wC))

Output()

## Where the plain pulse wins — telling two aircraft apart by speed

Two aircraft in close formation, a few tens of metres apart, 40 km away. No practical pulse resolves that in range: it would need $B=c/2\Delta R\approx15$ MHz just to split 10 m, and even then a formation flying line abreast sits in *one* range cell no matter what.

They are separated by **motion**. Transmit a coherent burst of $N$ plain pulses and look at how the echo phase advances from pulse to pulse; a target closing at $v_r$ turns that into a tone at $f_d=2v_r/\lambda$, and the burst behaves like a spectrum analyser of length $T=N/\text{PRF}$:

$$\Delta v=\frac{\lambda}{2T}=\frac{\lambda\,\text{PRF}}{2N}$$

This is exactly where the unmodulated pulse is the right tool. It has no modulation to smear the spectral line, so all its energy lands in one Doppler bin and the measurement is as clean as it can be. Verified in the panels: at $T=5$ ms the resolution is 3.0 m/s, and two targets 1 m/s apart merge into one line while 3 m/s apart shows two.

Drag `burst length` and watch the two lines fuse and split. Nothing about the *targets* changes — only how long you looked. Then note the cost in the readout: a long burst means a long dwell in one beam position, and every millisecond spent resolving these two is a millisecond not spent searching anywhere else.

In [3]:
def draw_doppler_scene(k, burst_ms, prf_khz, v1, dv, sep_m, tau_us):
    prf = prf_khz * 1e3
    N = max(int(burst_ms * 1e-3 * prf), 4)
    T = N / prf
    dv_res = LAM / (2 * T)
    v2 = v1 + dv
    R1 = 40e3 - v1 * k * 0.05
    R2 = R1 + sep_m
    fs = 20e6
    tau = tau_us * 1e-6
    ref = wf_cw(tau, fs)
    nsamp = int(fs * 2 * 600 / C0) + 3 * len(ref)
    rx = (echo(nsamp, ref, 2 * 300 / C0, fs, 1.0)
          + echo(nsamp, ref, 2 * (300 + sep_m) / C0, fs, 0.9))
    prof = np.abs(compress(rx, ref))
    rax = (np.arange(nsamp) - nsamp / 2) / fs * C0 / 2

    m = np.arange(N) / prf
    slow = (np.exp(2j * np.pi * 2 * v1 / LAM * m)
            + 0.9 * np.exp(2j * np.pi * 2 * v2 / LAM * m))
    nf = 4096
    D = np.abs(np.fft.fftshift(np.fft.fft(slow * np.hanning(N), nf)))
    fax = np.fft.fftshift(np.fft.fftfreq(nf, 1 / prf))
    vax = fax * LAM / 2

    fig = plt.figure(figsize=(13.2, 5.2))
    gs = fig.add_gridspec(2, 3, width_ratios=[1.05, 1.05, 0.55],
                          height_ratios=[1, 0.85], wspace=0.28, hspace=0.5,
                          left=0.055, right=0.995, top=0.90, bottom=0.11)

    a0 = panel(fig.add_subplot(gs[0, 0]), BLUE)
    a0.plot([0], [0], "^", ms=12, color="#f2f5fa")
    for R, v, col, nm in ((R1, v1, ORANGE, "A"), (R2, v2, GREEN, "B")):
        a0.plot(R / 1000, 0.6 if nm == "A" else -0.6, "o", ms=9, color=col)
        a0.annotate("", xy=(R / 1000 - v / 60, 0.6 if nm == "A" else -0.6),
                    xytext=(R / 1000, 0.6 if nm == "A" else -0.6),
                    arrowprops=dict(arrowstyle="-|>", color=col, lw=1.4))
        a0.text(R / 1000, (0.95 if nm == "A" else -1.25),
                f"{nm}  {v:.0f} m/s", color=col, fontsize=7, ha="center")
    a0.set_xlim(-2, 45); a0.set_ylim(-2.2, 2.2); a0.set_yticks([])
    a0.set_xlabel("range  (km)")
    a0.set_title(f"two aircraft {sep_m:.0f} m apart, closing at "
                 f"{v1:.0f} and {v2:.0f} m/s")

    a1 = panel(fig.add_subplot(gs[1, 0]), ORANGE)
    a1.plot(rax, dbv(prof), color=ORANGE, lw=1.1)
    a1.axvline(0, color=ORANGE, lw=0.7, ls=":")
    a1.axvline(sep_m, color=GREEN, lw=0.7, ls=":")
    a1.set_xlim(-3 * C0 * tau / 2, 3 * C0 * tau / 2); a1.set_ylim(-40, 4)
    a1.set_xlabel("range offset  (m)"); a1.set_ylabel("dB")
    a1.set_title(f"range: one blob — ΔR = {C0*tau/2:.0f} m, they are "
                 f"{sep_m:.0f} m apart")

    a2 = panel(fig.add_subplot(gs[:, 1]), GREEN)
    a2.plot(vax, dbv(D), color=GREEN, lw=1.2)
    a2.axvline(v1, color=ORANGE, lw=0.9, ls=":")
    a2.axvline(v2, color=CYAN, lw=0.9, ls=":")
    a2.set_xlim(v1 - 12 * max(dv_res, 1), v1 + 12 * max(dv_res, 1))
    a2.set_ylim(-45, 4)
    a2.set_xlabel("radial velocity  (m/s)"); a2.set_ylabel("dB")
    sep_ok = abs(dv) > dv_res
    a2.set_title(f"Doppler: {'two lines' if sep_ok else 'one line'} — "
                 f"Δv = {dv_res:.2f} m/s, targets {dv:.1f} m/s apart")

    readout(fig, 0.845, 0.90, [
        "BURST", "─" * 26,
        f"pulses N    {N:>10d}",
        f"PRF         {prf_khz:>10.2f}kHz",
        f"dwell T     {T*1e3:>10.2f}ms",
        f"τ           {tau_us:>10.2f}µs",
        "", "RESOLUTION", "─" * 26,
        f"Δv = λ/2T   {dv_res:>10.2f}m/s",
        f"ΔR = cτ/2   {C0*tau/2:>10.0f}m",
        f"target Δv   {dv:>10.2f}m/s",
        f"target ΔR   {sep_m:>10.0f}m",
        "", "VERDICT", "─" * 26,
        f"in range    {'merged':>10s}",
        f"in Doppler  {('SPLIT' if sep_ok else 'merged'):>10s}",
        "", f"v blind     {LAM*prf/2:>10.0f}m/s",
        f"R unamb     {C0/(2*prf)/1000:>10.1f}km",
        f"time cost   {T*1e3:>10.2f}ms/look",
    ], color=GREEN if sep_ok else ORANGE)
    footer(fig, f"coherent burst of {N} unmodulated pulses   PRF {prf_khz:.1f} kHz   "
                f"dwell {T*1e3:.2f} ms   λ={LAM*100:.0f} cm")
    plt.show()


_pD, _sD = timeline(120, step=2, desc="time step")
wD = dict(burst_ms=widgets.FloatSlider(value=5, min=0.3, max=25, step=0.1,
                                       description="burst length ms:", **SL),
          prf_khz=widgets.FloatSlider(value=10, min=2, max=30, step=1,
                                      description="PRF (kHz):", **SL),
          v1=widgets.FloatSlider(value=180, min=50, max=350, step=5,
                                 description="target A m/s:", **SL),
          dv=widgets.FloatSlider(value=6, min=0.2, max=40, step=0.2,
                                 description="speed gap m/s:", **SL),
          sep_m=widgets.FloatSlider(value=40, min=5, max=300, step=5,
                                    description="range gap m:", **SL),
          tau_us=widgets.FloatSlider(value=5, min=1, max=20, step=1,
                                     description="pulse τ (µs):", **SL),
          k=_sD)
display(widgets.VBox([widgets.HBox([wD["burst_ms"], wD["prf_khz"], wD["v1"]]),
                      widgets.HBox([wD["dv"], wD["sep_m"], wD["tau_us"]]),
                      widgets.HBox([_pD, _sD])]),
        widgets.interactive_output(draw_doppler_scene, wD))

Output()

## The linear FM chirp — bandwidth without shortening the pulse

Sweep the frequency linearly across the pulse and the deadlock breaks. Duration and bandwidth become independent knobs:

$$s(t)=\exp\!\left(j\pi\frac{B}{\tau}t^2\right),\qquad
f_{\text{inst}}(t)=\frac{B}{\tau}t,\qquad -\tfrac{\tau}{2}\le t\le\tfrac{\tau}{2}$$

The matched filter squeezes the whole pulse into a spike of width $\approx0.886/B$, so the range resolution is set by $B$ while the energy is still set by $\tau$. The **compression ratio is the time–bandwidth product** $B\tau$, and it is also the SNR gain: a 20 µs pulse with 20 MHz of sweep compresses 400:1, worth 26 dB, and resolves 7.5 m instead of 3 km.

The bill arrives in two parts. Sidelobes sit at $-13.3$ dB, because compressing a rectangular sweep is the same mathematics as a uniformly illuminated aperture — the identical $-13.3$ dB. Weighting the filter fixes it exactly as an aperture taper does: Hamming buys $-42.2$ dB and pays $1.67\times$ main-lobe width.

The second part is subtler and appears in the next section: because time and frequency are tied together inside the pulse, a Doppler shift is indistinguishable from a small time shift.

In [4]:
def draw_chirp(tau_us, B_mhz, weight, fs_mhz):
    fs = fs_mhz * 1e6
    tau = tau_us * 1e-6
    B = B_mhz * 1e6
    s = wf_chirp(tau, fs, B)
    n = len(s)
    t = (np.arange(n) / fs - tau / 2) * 1e6

    fig = plt.figure(figsize=(13.2, 4.6))
    gs = fig.add_gridspec(1, 4, width_ratios=[1, 1, 1, 0.55], wspace=0.3,
                          left=0.05, right=0.995, top=0.86, bottom=0.16)

    a0 = panel(fig.add_subplot(gs[0]), BLUE)
    a0.plot(t, s.real, color=BLUE, lw=0.6)
    a0.plot(t, np.abs(s), color="#f2f5fa", lw=1.0, ls="--")
    a0.set_xlabel("time  (µs)"); a0.set_ylabel("amplitude")
    a0.set_title("the sweep — frequency rises across the pulse")

    a1 = panel(fig.add_subplot(gs[1]), CYAN)
    a1.plot(t, (B / tau) * (t * 1e-6) / 1e6, color=CYAN, lw=1.6)
    a1.axhline(B / 2e6, color=GRIDC, lw=0.8, ls=":")
    a1.axhline(-B / 2e6, color=GRIDC, lw=0.8, ls=":")
    a1.set_xlabel("time  (µs)"); a1.set_ylabel("instantaneous f  (MHz)")
    a1.set_title(f"slope B/τ = {B/tau/1e12:.3f} MHz/µs")

    a2 = panel(fig.add_subplot(gs[2]), GREEN)
    ac = compress(np.pad(s, n), s, weight)
    rr = (np.arange(len(ac)) - len(ac) / 2) / fs * C0 / 2
    g = dbv(ac)
    a2.plot(rr, g, color=GREEN, lw=1.0)
    pk = int(np.argmax(g))
    loc = [g[i] for i in range(1, len(g) - 1)
           if g[i] > g[i - 1] and g[i] >= g[i + 1]
           and abs(rr[i] - rr[pk]) > 1.4 * C0 / (2 * B)]
    sll = max(loc) if loc else np.nan
    hp = rr[g >= -3.0]
    width = hp.max() - hp.min() if len(hp) > 1 else np.nan
    a2.axhline(sll, color=RED, lw=0.8, ls=":")
    a2.set_xlim(-14 * C0 / (2 * B), 14 * C0 / (2 * B)); a2.set_ylim(-60, 4)
    a2.set_xlabel("range offset  (m)"); a2.set_ylabel("compressed  (dB)")
    a2.set_title(f"compressed — {weight}, sidelobes {sll:.1f} dB")

    readout(fig, 0.845, 0.86, [
        "LFM CHIRP", "─" * 26,
        f"τ           {tau_us:>10.2f}µs",
        f"sweep B     {B_mhz:>10.2f}MHz",
        f"slope       {B/tau/1e12:>10.3f}MHz/µs",
        f"time-bw     {B*tau:>10.0f}",
        "", "COMPRESSION", "─" * 26,
        f"gain        {10*np.log10(B*tau):>10.2f}dB",
        f"width       {width:>10.2f}m",
        f"0.886c/2B   {0.886*C0/(2*B):>10.2f}m",
        f"ΔR = c/2B   {C0/(2*B):>10.2f}m",
        f"sidelobes   {sll:>10.2f}dB",
        f"filter      {weight:>10s}",
        "", "vs a plain pulse", "─" * 26,
        f"same τ, ΔR  {C0*tau/2:>10.0f}m",
        f"improvement {C0*tau/2/(C0/(2*B)):>10.0f}×",
    ])
    footer(fig, f"LFM   τ={tau_us:.1f} µs   B={B_mhz:.1f} MHz   Bτ={B*tau:.0f}   "
                f"matched filter {weight}")
    plt.show()


wF = dict(tau_us=widgets.FloatSlider(value=20, min=2, max=60, step=1,
                                     description="pulse τ (µs):", **SL),
          B_mhz=widgets.FloatSlider(value=10, min=1, max=50, step=1,
                                    description="sweep B (MHz):", **SL),
          weight=widgets.Dropdown(options=["unweighted", "hamming"],
                                  value="unweighted", description="MF weight:", **SL),
          fs_mhz=widgets.FloatSlider(value=160, min=80, max=400, step=20,
                                     description="sample rate:", **SL))
display(widgets.HBox([wF["tau_us"], wF["B_mhz"], wF["weight"], wF["fs_mhz"]]),
        widgets.interactive_output(draw_chirp, wF))

Output()

## Where the chirp wins — two targets in one range cell, 80 km out

A fighter and the tanker it is refuelling from, 30 m apart at 80 km. Detecting anything at that range needs a long pulse for energy; separating them needs a short one. The chirp is the only waveform here that provides both.

The two range profiles are computed from the **same pulse length** and the same energy, so the comparison is honest: only the modulation differs. The plain pulse merges them into a single return. The chirp splits them the moment their separation exceeds $c/2B$ — measured merged at 7.5 m and split at 40 m for $B=10$ MHz, where the theory says $\Delta R=15.0$ m.

Now the catch, and it is the reason chirps are not free. Inside the pulse, time and frequency are the same variable, so the matched filter cannot distinguish a genuine delay from a Doppler shift. A moving target is reported at the wrong range:

$$\Delta R_{\text{coupling}}=-\frac{c\,f_d\,\tau}{2B}$$

Verified against the simulation to within 0.023 m: a target closing at 300 m/s with $\tau=20$ µs and $B=10$ MHz reads **6 m closer than it is**. Increase $\tau$ or decrease $B$ and the error grows in proportion. It is a bias, not noise — it does not average away, and a radar that cares about absolute range either measures Doppler separately and corrects, or alternates up-sweeps and down-sweeps so the two errors cancel.

In [ ]:
def draw_chirp_scene(k, tau_us, B_mhz, sep_m, vr, weight):
    fs = 160e6
    tau, B = tau_us * 1e-6, B_mhz * 1e6
    R0 = 80e3 - vr * k * 0.02
    ch = wf_chirp(tau, fs, B)
    cw = wf_cw(tau, fs)
    span = 400.0
    nsamp = int(2 * span / C0 * fs) + 3 * len(ch)
    fd = 2 * vr / LAM
    d0 = span / 2

    out = {}
    for nm, ref in (("chirp", ch), ("plain pulse", cw)):
        rx = (echo(nsamp, ref, 2 * d0 / C0, fs, 1.0, fd)
              + echo(nsamp, ref, 2 * (d0 + sep_m) / C0, fs, 0.85, fd))
        out[nm] = np.abs(compress(rx, ref, weight if nm == "chirp" else "unweighted"))
    rax = (np.arange(nsamp) - nsamp / 2) / fs * C0 / 2

    def peak_m(sig):
        i = int(np.argmax(sig))
        a, b, c = sig[i - 1], sig[i], sig[i + 1]
        d = 0.5 * (a - c) / (a - 2 * b + c) if (a - 2 * b + c) != 0 else 0.0
        return ((i + d) - nsamp / 2) / fs * C0 / 2

    # register against the zero-Doppler peak so the readout shows the physics,
    # not the sub-sample offset of the discrete correlation
    # the coupling is a property of ONE echo: measuring it on the two-target
    # profile would report the blend of both peaks instead
    ref_pos = peak_m(np.abs(compress(echo(nsamp, ch, 2 * d0 / C0, fs, 1.0, 0.0),
                                     ch, weight)))
    meas = peak_m(np.abs(compress(echo(nsamp, ch, 2 * d0 / C0, fs, 1.0, fd),
                                  ch, weight)))
    shift = meas - ref_pos
    pred = -C0 * fd * tau / (2 * B)

    fig = plt.figure(figsize=(13.2, 5.2))
    gs = fig.add_gridspec(2, 3, width_ratios=[1.05, 1.05, 0.55],
                          wspace=0.28, hspace=0.5, left=0.055, right=0.995,
                          top=0.90, bottom=0.11)

    a0 = panel(fig.add_subplot(gs[0, 0]), BLUE)
    a0.plot([0], [0], "^", ms=12, color="#f2f5fa")
    a0.plot(R0 / 1000, 0.5, "o", ms=10, color=ORANGE)
    a0.plot((R0 + sep_m) / 1000, -0.5, "o", ms=7, color=GREEN)
    a0.annotate("", xy=(R0 / 1000 - vr / 120, 0.5), xytext=(R0 / 1000, 0.5),
                arrowprops=dict(arrowstyle="-|>", color=ORANGE, lw=1.4))
    a0.text(R0 / 1000, 1.0, f"pair {sep_m:.0f} m apart, {vr:.0f} m/s",
            color=FG, fontsize=7, ha="center")
    a0.set_xlim(-3, 95); a0.set_ylim(-2, 2); a0.set_yticks([])
    a0.set_xlabel("range  (km)")
    a0.set_title("same pulse length, same energy — only the modulation differs")

    a1 = panel(fig.add_subplot(gs[1, 0]), ORANGE)
    a1.plot(rax - ref_pos, dbv(out["plain pulse"]), color=ORANGE, lw=1.0)
    a1.axvline(0, color="#f2f5fa", lw=0.7, ls=":")
    a1.axvline(sep_m, color=GREEN, lw=0.7, ls=":")
    a1.set_xlim(-2.5 * sep_m - 40, 2.5 * sep_m + 40); a1.set_ylim(-40, 4)
    a1.set_xlabel("range offset  (m)"); a1.set_ylabel("dB")
    a1.set_title(f"plain pulse — ΔR = {C0*tau/2:.0f} m: one target")

    a2 = panel(fig.add_subplot(gs[:, 1]), GREEN)
    a2.plot(rax - ref_pos, dbv(out["chirp"]), color=GREEN, lw=1.1)
    a2.axvline(0, color="#f2f5fa", lw=0.8, ls=":")
    a2.axvline(sep_m, color=CYAN, lw=0.8, ls=":")
    a2.axvline(shift, color=RED, lw=1.0)
    a2.set_xlim(-2.5 * sep_m - 40, 2.5 * sep_m + 40); a2.set_ylim(-45, 4)
    a2.set_xlabel("range offset  (m)"); a2.set_ylabel("dB")
    ok = sep_m > C0 / (2 * B)
    a2.set_title(f"chirp — ΔR = {C0/(2*B):.1f} m: "
                 f"{'two targets' if ok else 'still one'}   |   red line = "
                 f"where Doppler moved the peak")

    readout(fig, 0.845, 0.90, [
        "WAVEFORM", "─" * 26,
        f"τ           {tau_us:>10.1f}µs",
        f"B           {B_mhz:>10.1f}MHz",
        f"Bτ          {B*tau:>10.0f}",
        f"filter      {weight:>10s}",
        "", "RESOLUTION", "─" * 26,
        f"plain ΔR    {C0*tau/2:>10.0f}m",
        f"chirp ΔR    {C0/(2*B):>10.2f}m",
        f"target gap  {sep_m:>10.0f}m",
        f"chirp says  {'TWO' if ok else 'ONE':>10s}",
        "", "DOPPLER COUPLING", "─" * 26,
        f"v_r         {vr:>+10.0f}m/s",
        f"f_d         {fd/1e3:>+10.2f}kHz",
        f"measured    {shift:>+10.2f}m",
        f"-c·fd·τ/2B  {pred:>+10.2f}m",
        f"error       {abs(shift-pred):>10.3f}m",
        "range bias, not noise",
    ], color=RED if abs(pred) > C0 / (2 * B) else FG)
    footer(fig, f"τ={tau_us:.0f} µs   B={B_mhz:.0f} MHz   Bτ={B*tau:.0f}   "
                f"coupling ΔR = −c·fd·τ/2B   λ={LAM*100:.0f} cm")
    plt.show()


_pF, _sF = timeline(80, step=2, desc="time step")
wS = dict(tau_us=widgets.FloatSlider(value=20, min=5, max=50, step=1,
                                     description="pulse τ (µs):", **SL),
          B_mhz=widgets.FloatSlider(value=10, min=1, max=40, step=1,
                                    description="sweep B (MHz):", **SL),
          sep_m=widgets.FloatSlider(value=40, min=5, max=200, step=5,
                                    description="target gap m:", **SL),
          vr=widgets.FloatSlider(value=0, min=-600, max=600, step=25,
                                 description="closing v (m/s):", **SL),
          weight=widgets.Dropdown(options=["unweighted", "hamming"],
                                  value="unweighted", description="MF weight:", **SL),
          k=_sF)
display(widgets.VBox([widgets.HBox([wS["tau_us"], wS["B_mhz"], wS["sep_m"]]),
                      widgets.HBox([wS["vr"], wS["weight"]]),
                      widgets.HBox([_pF, _sF])]),
        widgets.interactive_output(draw_chirp_scene, wS))

Output()

## The phase-coded pulse — compression with a switch instead of a sweep

Divide the pulse into $N$ equal chips and flip the phase by 180° according to a fixed code. No frequency sweep is needed, so the transmitter only has to switch a sign — historically far cheaper than generating a linear ramp, and still attractive in low-cost and digital-first designs.

$$s(t)=\sum_{n=0}^{N-1}c_n\,\text{rect}\!\left(\frac{t-nt_c}{t_c}\right),\qquad c_n\in\{+1,-1\}$$

Bandwidth is set by the **chip** rather than the pulse, $B\approx1/t_c$, so a 13-chip code compresses 13:1 and the range resolution improves by the same factor. The Barker codes are the special ones whose autocorrelation sidelobes are all exactly $1/N$ — the best possible for a biphase code, and Barker-13 is the longest that exists.

That is also the ceiling. The peak-to-sidelobe ratio is fixed at $1/13=-22.28$ dB, measured exactly in the panel. Unlike a chirp, **you cannot weight it away**: the sidelobes come from the code itself, not from a rectangular envelope, so there is no taper to apply. Longer codes exist — combined and polyphase — but no biphase code beats $-22.3$ dB at any length.

Compare the compressed outputs across the three waveforms and the trade is plain: the plain pulse has no sidelobes and no resolution, the chirp has resolution and adjustable sidelobes, the Barker code has resolution and a floor you are stuck with.

In [6]:
CODES = {"Barker-13": BARKER13,
         "Barker-7": np.array([1, 1, 1, -1, -1, 1, -1], float),
         "Barker-5": np.array([1, 1, 1, -1, 1], float),
         "random-13": np.sign(np.random.default_rng(3).normal(size=13))}


def draw_barker(code_name, tau_us, fs_mhz):
    fs = fs_mhz * 1e6
    tau = tau_us * 1e-6
    code = CODES[code_name]
    N = len(code)
    s = wf_barker(tau, fs, code)
    n = len(s)
    tc = tau / N
    B = 1.0 / tc
    t = np.arange(n) / fs * 1e6

    fig = plt.figure(figsize=(13.2, 4.6))
    gs = fig.add_gridspec(1, 4, width_ratios=[1, 1, 1, 0.55], wspace=0.3,
                          left=0.05, right=0.995, top=0.86, bottom=0.16)

    a0 = panel(fig.add_subplot(gs[0]), BLUE)
    a0.step(t, s.real, where="post", color=BLUE, lw=1.3)
    for i in range(N + 1):
        a0.axvline(i * tc * 1e6, color=GRIDC, lw=0.6)
    a0.set_ylim(-1.6, 1.6); a0.set_xlabel("time  (µs)")
    a0.set_ylabel("phase  (±1)")
    a0.set_title(f"{code_name} — {N} chips of {tc*1e6:.3f} µs")

    a1 = panel(fig.add_subplot(gs[1]), ORANGE)
    nf = 1 << 14
    S = np.abs(np.fft.fftshift(np.fft.fft(s, nf)))
    f = np.fft.fftshift(np.fft.fftfreq(nf, 1 / fs)) / 1e6
    a1.plot(f, dbv(S), color=ORANGE, lw=0.9)
    a1.axvline(B / 2e6, color=CYAN, lw=0.8, ls=":")
    a1.axvline(-B / 2e6, color=CYAN, lw=0.8, ls=":")
    a1.set_xlim(-4 * B / 1e6, 4 * B / 1e6); a1.set_ylim(-45, 4)
    a1.set_xlabel("frequency  (MHz)"); a1.set_ylabel("dB")
    a1.set_title(f"the chip sets the bandwidth — B ≈ 1/t_c = {B/1e6:.2f} MHz")

    a2 = panel(fig.add_subplot(gs[2]), GREEN)
    ac = compress(np.pad(s, n), s)
    rr = (np.arange(len(ac)) - len(ac) / 2) / fs * C0 / 2
    g = dbv(ac)
    pk = int(np.argmax(g))
    chip_m = C0 * tc / 2
    loc = [g[i] for i in range(1, len(g) - 1)
           if g[i] > g[i - 1] and g[i] >= g[i + 1]
           and abs(rr[i] - rr[pk]) > 1.2 * chip_m]
    psl = max(loc) if loc else np.nan
    a2.plot(rr, g, color=GREEN, lw=1.0)
    a2.axhline(psl, color=RED, lw=1.0, ls="--")
    a2.text(rr.max() * 0.98, psl + 1.5, f"{psl:.2f} dB floor", color=RED,
            fontsize=7, ha="right")
    a2.set_xlim(-1.3 * C0 * tau / 2, 1.3 * C0 * tau / 2); a2.set_ylim(-45, 4)
    a2.set_xlabel("range offset  (m)"); a2.set_ylabel("dB")
    a2.set_title("compressed — the sidelobe floor is set by the code")

    readout(fig, 0.845, 0.86, [
        "PHASE CODE", "─" * 26,
        f"code        {code_name:>14s}",
        f"chips N     {N:>10d}",
        f"τ           {tau_us:>10.2f}µs",
        f"chip t_c    {tc*1e6:>10.3f}µs",
        f"bandwidth   {B/1e6:>10.2f}MHz",
        "", "COMPRESSION", "─" * 26,
        f"ratio       {N:>10d}:1",
        f"gain        {10*np.log10(N):>10.2f}dB",
        f"ΔR = c/2B   {C0/(2*B):>10.1f}m",
        f"measured    {psl:>10.2f}dB",
        f"1/N ideal   {20*np.log10(1/N):>10.2f}dB",
        "", "cannot be weighted away",
        "Barker-13 is the longest",
        "biphase code that exists",
    ], color=GREEN if abs(psl - 20 * np.log10(1 / N)) < 1.0 else ORANGE)
    footer(fig, f"{code_name}   {N} chips   t_c={tc*1e6:.3f} µs   B={B/1e6:.2f} MHz   "
                f"PSL {psl:.2f} dB")
    plt.show()


wK = dict(code_name=widgets.Dropdown(options=list(CODES), value="Barker-13",
                                     description="code:", **SL),
          tau_us=widgets.FloatSlider(value=13, min=2.6, max=52, step=1.3,
                                     description="pulse τ (µs):", **SL),
          fs_mhz=widgets.FloatSlider(value=80, min=40, max=240, step=20,
                                     description="sample rate:", **SL))
display(widgets.HBox([wK["code_name"], wK["tau_us"], wK["fs_mhz"]]),
        widgets.interactive_output(draw_barker, wK))

Output()

## Where the phase code fails — a small target beside a large one

A tanker and a drone. The tanker's echo is 25 dB stronger and sits a few range cells away. The question is not whether the radar can *resolve* them — both waveforms have the resolution — but whether the small return survives the large one's range sidelobes.

The Barker code's floor is fixed at $-22.28$ dB, so the arithmetic is brutally simple: a target more than 22.3 dB below its neighbour lands underneath the sidelobes and disappears. Verified across levels — a return at $-18$ dB is visible, one at $-28$ dB is gone, and the crossover sits exactly at the code's own floor. No amount of SNR helps, because the sidelobes scale with the interferer, not with the noise.

The chirp is not intrinsically better — unweighted it is $-13.3$ dB, which is *worse*. What it has is a **knob**. Hamming-weighting the matched filter drops the floor to $-42.2$ dB for $1.67\times$ the main-lobe width, and the drone reappears. The phase code has no equivalent move: its sidelobes are a property of the sequence.

Drag the drone's range past the tanker and watch it vanish and return as it crosses the sidelobe region — then switch the waveform and watch the same geometry give a different answer. This is what "choosing a waveform" actually means: not picking the best one, but knowing which failure you can afford.

In [7]:
def draw_masking(k, wf, weight, rel_db, gap_m, tau_us, B_mhz):
    fs = 160e6
    tau = tau_us * 1e-6
    if wf == "Barker-13":
        ref = wf_barker(tau, fs)
        B = len(BARKER13) / tau
        floor_db = 20 * np.log10(1 / 13)
    else:
        B = B_mhz * 1e6
        ref = wf_chirp(tau, fs, B)
        floor_db = -13.3 if weight == "unweighted" else -42.2
    drift = 40.0 * np.sin(2 * np.pi * k / 80)
    gap = gap_m + drift
    span = 900.0
    nsamp = int(2 * span / C0 * fs) + 3 * len(ref)
    d0 = span / 3
    rx = (echo(nsamp, ref, 2 * d0 / C0, fs, 1.0)
          + echo(nsamp, ref, 2 * (d0 + gap) / C0, fs, 10 ** (rel_db / 20)))
    y = np.abs(compress(rx, ref, weight))
    g = dbv(y, -70)
    rax = (np.arange(nsamp) - nsamp / 2) / fs * C0 / 2
    ref_pos = ((int(round(2 * d0 / C0 * fs)) + len(ref) // 2) - nsamp / 2) / fs * C0 / 2
    x = rax - ref_pos
    j = int(np.argmin(np.abs(x - gap)))
    lvl = g[max(j - 3, 0):j + 4].max()
    visible = rel_db > floor_db + 2

    fig = plt.figure(figsize=(13.2, 5.2))
    gs = fig.add_gridspec(2, 3, width_ratios=[1.05, 1.05, 0.55],
                          wspace=0.28, hspace=0.5, left=0.055, right=0.995,
                          top=0.90, bottom=0.11)

    a0 = panel(fig.add_subplot(gs[0, 0]), BLUE)
    a0.plot([0], [0], "^", ms=12, color="#f2f5fa")
    a0.plot(60, 0.6, "o", ms=13, color=ORANGE)
    a0.text(60, 1.15, "tanker  0 dB", color=ORANGE, fontsize=7, ha="center")
    a0.plot(60 + gap / 1000, -0.6, "o", ms=5,
            color=GREEN if visible else RED)
    a0.text(60 + gap / 1000, -1.35, f"drone  {rel_db:+.0f} dB", fontsize=7,
            ha="center", color=GREEN if visible else RED)
    a0.set_xlim(-3, 80); a0.set_ylim(-2.2, 2.2); a0.set_yticks([])
    a0.set_xlabel("range  (km)")
    a0.set_title(f"{gap:.0f} m apart — the question is sidelobes, not resolution")

    a1 = panel(fig.add_subplot(gs[:, 1]), GREEN if visible else RED)
    a1.plot(x, g, color=GREEN if visible else RED, lw=1.1)
    a1.axhline(floor_db, color=CYAN, lw=1.0, ls="--")
    a1.text(x.max() * 0.98, floor_db + 1.6, f"{wf} floor {floor_db:.1f} dB",
            color=CYAN, fontsize=7, ha="right")
    a1.axvline(gap, color="#f2f5fa", lw=0.9, ls=":")
    a1.plot([gap], [lvl], "o", ms=8, color=GREEN if visible else RED)
    a1.set_xlim(-260, 320); a1.set_ylim(-60, 4)
    a1.set_xlabel("range offset from the tanker  (m)"); a1.set_ylabel("dB")
    a1.set_title(f"{'drone stands clear' if visible else 'drone is under the sidelobes'}")

    a2 = panel(fig.add_subplot(gs[1, 0]), ORANGE)
    lv = np.linspace(-45, 0, 200)
    a2.fill_between(lv, -70, np.where(lv > floor_db, 0, -70),
                    color=GREEN, alpha=0.16)
    a2.axvline(floor_db, color=CYAN, lw=1.2)
    a2.axvline(rel_db, color="#f2f5fa", lw=1.2)
    a2.set_xlim(-45, 0); a2.set_ylim(-70, 2); a2.set_yticks([])
    a2.set_xlabel("target level relative to the big one  (dB)")
    a2.set_title("everything left of the line is invisible")

    readout(fig, 0.845, 0.90, [
        "WAVEFORM", "─" * 26,
        f"{wf:>26s}",
        f"filter      {weight:>10s}",
        f"τ           {tau_us:>10.1f}µs",
        f"bandwidth   {B/1e6:>10.2f}MHz",
        f"ΔR          {C0/(2*B):>10.1f}m",
        "", "SIDELOBES", "─" * 26,
        f"floor       {floor_db:>10.2f}dB",
        f"weightable  {('no' if wf=='Barker-13' else 'yes'):>10s}",
        "", "SCENE", "─" * 26,
        f"gap         {gap:>10.0f}m",
        f"drone level {rel_db:>+10.1f}dB",
        f"margin      {rel_db-floor_db:>+10.1f}dB",
        "DETECTED" if visible else "MASKED",
    ], color=GREEN if visible else RED)
    footer(fig, f"{wf}   {weight} filter   sidelobe floor {floor_db:.1f} dB   "
                f"drone {rel_db:+.0f} dB   gap {gap:.0f} m")
    plt.show()


_pM, _sM = timeline(79, desc="time step")
wM = dict(wf=widgets.Dropdown(options=["Barker-13", "LFM chirp"],
                              value="Barker-13", description="waveform:", **SL),
          weight=widgets.Dropdown(options=["unweighted", "hamming"],
                                  value="unweighted", description="MF weight:", **SL),
          rel_db=widgets.FloatSlider(value=-25, min=-45, max=-2, step=1,
                                     description="drone level dB:", **SL),
          gap_m=widgets.FloatSlider(value=120, min=30, max=260, step=10,
                                    description="range gap m:", **SL),
          tau_us=widgets.FloatSlider(value=13, min=6.5, max=39, step=6.5,
                                     description="pulse τ (µs):", **SL),
          B_mhz=widgets.FloatSlider(value=10, min=2, max=40, step=1,
                                    description="chirp B (MHz):", **SL),
          k=_sM)
display(widgets.VBox([widgets.HBox([wM["wf"], wM["weight"], wM["rel_db"]]),
                      widgets.HBox([wM["gap_m"], wM["tau_us"], wM["B_mhz"]]),
                      widgets.HBox([_pM, _sM])]),
        widgets.interactive_output(draw_masking, wM))

Output()